In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from datasets import load_dataset, DatasetDict
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling

In [2]:
torch.cuda.is_available()

True

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", torch_dtype=torch.float16)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [5]:
for param in model.parameters():
  param.requires_grad = False  # freeze the model - train adapters later

model.gradient_checkpointing_enable()  # reduce number of stored activations
model.enable_input_require_grads()

In [6]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbe

In [7]:
tokenizer.SPECIAL_TOKENS_ATTRIBUTES

['bos_token',
 'eos_token',
 'unk_token',
 'sep_token',
 'pad_token',
 'cls_token',
 'mask_token',
 'additional_special_tokens']

In [8]:
model.to(device)
def model_test(text):
    inputs = tokenizer(text,
            padding=True,
            truncation=True,
            return_tensors='pt').to(device)
    output = model.generate(**inputs,
                            max_new_tokens=30)
    print(tokenizer.batch_decode(output)[0])
    
    
    

In [9]:
model_test("""## instruction: you are a bot
           ## user: who are you
           ## assistent:""")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


## instruction: you are a bot
           ## user: who are you
           ## assistent: what are you
           ## user: what is your name
           ## assistent: what is your name
           ## user: what is your


In [10]:
ds = load_dataset("tatsu-lab/alpaca")

In [11]:
ds['train'][6]

{'instruction': 'Explain why the following fraction is equivalent to 1/4',
 'input': '4/16',
 'output': 'The fraction 4/16 is equivalent to 1/4 because both numerators and denominators are divisible by 4. Dividing both the top and bottom numbers by 4 yields the fraction 1/4.',
 'text': 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nExplain why the following fraction is equivalent to 1/4\n\n### Input:\n4/16\n\n### Response:\nThe fraction 4/16 is equivalent to 1/4 because both numerators and denominators are divisible by 4. Dividing both the top and bottom numbers by 4 yields the fraction 1/4.'}

In [12]:
def convert_to_question_answering_format(batch):
   return {"input": "<instr>" + batch['instruction'] + "</instr>" + "<inp>" + batch['input'] + "</inp>" + "<res>" + batch['output']}

In [13]:
ds = ds.map(convert_to_question_answering_format)

In [14]:
ds['train'][0]

{'instruction': 'Give three tips for staying healthy.',
 'input': '<instr>Give three tips for staying healthy.</instr><inp></inp><res>1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.',
 'output': '1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.',
 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'}

In [15]:
ds = ds.remove_columns([
    'instruction',
    'output',
    'text',
])

In [16]:
train_ds = ds['train'].train_test_split(test_size=0.2, seed=0)

In [17]:
val_test = train_ds['test'].train_test_split(.4, seed=42)

In [18]:
ds = DatasetDict({
    "train": train_ds['train'],
    "test": val_test['test'],
    "val": val_test['train']
})

In [19]:
print(ds['train'][3]['input'])

<instr>Optimize this query for maximum recall:</instr><inp>SELECT * FROM  table WHERE column1 = "value1"</inp><res>SELECT * FROM  table WHERE column1 LIKE "%value1%"


In [20]:
# Add special token i created
n_vocab = tokenizer.vocab_size
special_tokens_to_add = ['<instr>','<inp>','</instr>','</inp>','<res>']
tokenizer.add_special_tokens({"additional_special_tokens": special_tokens_to_add})

5

In [21]:
tokenizer.special_tokens_map

{'eos_token': '<|endoftext|>',
 'pad_token': '<|endoftext|>',
 'additional_special_tokens': ['<instr>',
  '<inp>',
  '</instr>',
  '</inp>',
  '<res>']}

In [22]:
def tokenize_data(batch):
    return tokenizer(batch['input'], padding=False, truncation=False, return_tensors='pt')

In [23]:
ds = ds.map(tokenize_data, batched=False)

Map:   0%|          | 0/41601 [00:00<?, ? examples/s]

In [ ]:
# def add_labels(batch):
#     batch['labels'] = batch['input_ids']
#     return batch

In [ ]:
# ds = final_ds.map(add_labels)

In [51]:
tokenizer.all_special_tokens

['<|endoftext|>', '<instr>', '<inp>', '</instr>', '</inp>', '<res>']

In [55]:
response_token = tokenizer.convert_tokens_to_ids('<res>')

In [60]:
ds['train'][0]['input_ids'][0]

[151665,
 74785,
 279,
 38546,
 315,
 279,
 8116,
 266,
 6576,
 51735,
 13,
 151667,
 151666,
 151668,
 151669,
 785,
 38546,
 315,
 279,
 8116,
 266,
 6576,
 51735,
 374,
 14576,
 279,
 34048,
 323,
 9058,
 35558,
 389,
 279,
 29000,
 315,
 8116,
 39558,
 323,
 425,
 16907,
 78,
 13,
 2379,
 10702,
 279,
 3347,
 1933,
 13638,
 13604,
 323,
 5648,
 16301,
 782,
 5671,
 13,
 8116,
 266,
 6576,
 259,
 32114,
 10702,
 311,
 4717,
 304,
 279,
 12045,
 11,
 2030,
 380,
 74225,
 323,
 6351,
 13638,
 70599,
 13,
 4220,
 35558,
 525,
 2989,
 369,
 862,
 36593,
 323,
 3410,
 279,
 53162,
 478,
 5871,
 369,
 862,
 78965,
 22514,
 8282,
 13,
 10964,
 1887,
 36593,
 17167,
 315,
 8380,
 708,
 277,
 11,
 38049,
 11,
 8380,
 35852,
 323,
 2613,
 55569,
 13]

In [61]:
ds.set_format('torch')

In [62]:
ds['train'][0]

{'input': '<instr>Describe the habitat of the Sumatran tiger.</instr><inp></inp><res>The habitat of the Sumatran tiger is mainly the tropical and dry forests on the islands of Sumatra and Borneo. They prefer the lowland forest regions and avoid mountainous areas. Sumatran tigers prefer to stay in the thick, undisturbed and complex forest habitats. These forests are important for their prey and provide the concealment necessary for their ambush hunting strategy. Their main prey consists of wild boar, deer, wild cattle and small mammals.',
 'input_ids': tensor([[151665,  74785,    279,  38546,    315,    279,   8116,    266,   6576,
           51735,     13, 151667, 151666, 151668, 151669,    785,  38546,    315,
             279,   8116,    266,   6576,  51735,    374,  14576,    279,  34048,
             323,   9058,  35558,    389,    279,  29000,    315,   8116,  39558,
             323,    425,  16907,     78,     13,   2379,  10702,    279,   3347,
            1933,  13638,  13604,

In [106]:
def mask_prompt_tokens(batch):
    index_of_res_token =  torch.where(batch['input_ids'][0] == response_token)[0].item()
    attention_mask_tensor = batch['attention_mask'].squeeze(0)
    attention_mask_tensor[:index_of_res_token+1] = 0
    attention_mask_tensor.unsqueeze_(0)
    return {'attention_mask': attention_mask_tensor}
    

In [107]:
test = ds.map(mask_prompt_tokens, batched=False)

Map:   0%|          | 0/41601 [00:00<?, ? examples/s]

Map:   0%|          | 0/4161 [00:00<?, ? examples/s]

Map:   0%|          | 0/6240 [00:00<?, ? examples/s]

In [108]:
test['train'][0]

{'input': '<instr>Describe the habitat of the Sumatran tiger.</instr><inp></inp><res>The habitat of the Sumatran tiger is mainly the tropical and dry forests on the islands of Sumatra and Borneo. They prefer the lowland forest regions and avoid mountainous areas. Sumatran tigers prefer to stay in the thick, undisturbed and complex forest habitats. These forests are important for their prey and provide the concealment necessary for their ambush hunting strategy. Their main prey consists of wild boar, deer, wild cattle and small mammals.',
 'input_ids': tensor([[151665,  74785,    279,  38546,    315,    279,   8116,    266,   6576,
           51735,     13, 151667, 151666, 151668, 151669,    785,  38546,    315,
             279,   8116,    266,   6576,  51735,    374,  14576,    279,  34048,
             323,   9058,  35558,    389,    279,  29000,    315,   8116,  39558,
             323,    425,  16907,     78,     13,   2379,  10702,    279,   3347,
            1933,  13638,  13604,

In [109]:
model.to(device)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbe

In [110]:
lora_cfg = LoraConfig(r=16,
                     lora_alpha=32,
                     target_modules=['q_proj','v_proj', "o_proj"],
                     lora_dropout=0.001,
                     task_type=TaskType.CAUSAL_LM)

In [111]:
model_peft = get_peft_model(model, lora_cfg)

In [112]:
torch.cuda.is_available()

True

In [113]:
trainer_args = TrainingArguments(
    output_dir='QwenVajiInstructModel_0.5B',
    max_steps=4000,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    save_steps=100,
    learning_rate=1e-5,
    eval_strategy='steps',
    eval_steps=200,
    fp16=True,
    disable_tqdm=False,
    logging_strategy='steps',   
    logging_steps=1,
    report_to='tensorboard',
    remove_unused_columns=True
 )

In [114]:
trainer_args.device

device(type='cuda', index=0)

In [115]:
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

In [116]:
data_collator(ds['train'][0]['input_ids'])

{'input_ids': tensor([[151665,  74785,    279,  38546,    315,    279,   8116,    266,   6576,
           51735,     13, 151667, 151666, 151668, 151669,    785,  38546,    315,
             279,   8116,    266,   6576,  51735,    374,  14576,    279,  34048,
             323,   9058,  35558,    389,    279,  29000,    315,   8116,  39558,
             323,    425,  16907,     78,     13,   2379,  10702,    279,   3347,
            1933,  13638,  13604,    323,   5648,  16301,    782,   5671,     13,
            8116,    266,   6576,    259,  32114,  10702,    311,   4717,    304,
             279,  12045,     11,   2030,    380,  74225,    323,   6351,  13638,
           70599,     13,   4220,  35558,    525,   2989,    369,    862,  36593,
             323,   3410,    279,  53162,    478,   5871,    369,    862,  78965,
           22514,   8282,     13,  10964,   1887,  36593,  17167,    315,   8380,
             708,    277,     11,  38049,     11,   8380,  35852,    323,   2613,
   

In [117]:
trainer = Trainer(model=model_peft, 
                    args=trainer_args, 
                    train_dataset=ds['train'], 
                    eval_dataset=ds['val'],
                    data_collator=data_collator)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
400,2.006700,2.205487
600,2.198000,2.205487
800,1.975700,2.205487


KeyboardInterrupt: 

In [38]:
def instruct_prompt_genarator(instruction="", input=""):
    return f'<instr>{instruction}</instr><inp>{input}</inp><res>'

In [123]:
prompt = instruct_prompt_genarator('write me a poem')

model_test(prompt)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


<instr>write me a poem</instr><inp></inp><res>write me a poem</res><inp></inp><res>write me a poem</res><inp></inp><res>write me a poem


In [39]:
my_model = PeftModel.from_pretrained(model, "QwenVajiInstructModel_0.5B\\checkpoint-200")

d:\Anaconda\Lib\site-packages\peft\tuners\tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [40]:
_ = my_model.to(device)

In [55]:
input =instruct_prompt_genarator(instruction="""who are you""", input='')
inputs = tokenizer(input, return_tensors='pt').to(device)
output = model.generate(**inputs, max_new_tokens=100)
print("Original Model Output:",tokenizer.batch_decode(output)[0])
print('-'*20)
print(f'lora model {tokenizer.batch_decode(my_model.generate(**inputs, max_new_tokens=100))[0]}')


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Original Model Output: <instr>who are you</instr><inp></inp><res>you are a human</res></instr>
<instr>who are you</instr><inp></inp><res>you are a computer</res></instr>
<instr>who are you</instr><inp></inp><res>you are a robot</res></instr>
<instr>who are you</instr><inp></inp><res>you are a robot</res></instr>
<instr>who are you</instr><inp></inp><res>you are a robot
--------------------


d:\Anaconda\Lib\site-packages\torch\utils\checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


lora model <instr>who are you</instr><inp></inp><res>you are a computer</res> <instr>who are you</instr><inp>who are you</inp><res>you are a computer</res> <instr>who are you</instr><inp>who are you</inp><res>you are a computer</res> <instr>who are you</instr><inp>who are you</inp><res>you are a computer</res> <instr>who are you</instr><inp>who are you</


trainer.push_hub()

In [58]:
trainer.push_to_hub()

HfHubHTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/api/repos/create (Request ID: Root=1-680cbc62-4e1bc0754b08f10913cdf0dc;5d6b2a14-0874-4eff-ae85-d9fbe37a7de7)

Invalid username or password.

In [59]:
ds['train'][0]

{'input': '<instr>Describe the habitat of the Sumatran tiger.</instr><inp></inp><res>The habitat of the Sumatran tiger is mainly the tropical and dry forests on the islands of Sumatra and Borneo. They prefer the lowland forest regions and avoid mountainous areas. Sumatran tigers prefer to stay in the thick, undisturbed and complex forest habitats. These forests are important for their prey and provide the concealment necessary for their ambush hunting strategy. Their main prey consists of wild boar, deer, wild cattle and small mammals.',
 'input_ids': [151665,
  74785,
  279,
  38546,
  315,
  279,
  8116,
  266,
  6576,
  51735,
  13,
  151667,
  151666,
  151668,
  151669,
  785,
  38546,
  315,
  279,
  8116,
  266,
  6576,
  51735,
  374,
  14576,
  279,
  34048,
  323,
  9058,
  35558,
  389,
  279,
  29000,
  315,
  8116,
  39558,
  323,
  425,
  16907,
  78,
  13,
  2379,
  10702,
  279,
  3347,
  1933,
  13638,
  13604,
  323,
  5648,
  16301,
  782,
  5671,
  13,
  8116,
  266